# Nutrition5k: DINOv2-L y DINOv3-L (pod)

Backbones grandes como referencia, no finalistas (ver `notes/decisions_and_concepts/reference-models.md`).

- **DINOv2-L y DINOv3-L** (~300 M, 14× `dinov2_s`) con la receta común (CLS ⊕ media, 224 px), semilla 0, con 4
  profundidades: `all`, 12, 6 y 0 de 24 bloques (100 / 50 / 25 / 0 % del backbone). En `dinov2_s` entrenar todo fue
  peor que la mitad y congelarlo todo perdió poco, así que con 14× más parámetros se mira por debajo de la mitad.
  Después, las semillas 1 y 2 del mejor nivel.
- GPU: el fine-tuning completo de un ViT-L en fp32 con batch 32 necesita ~17–20 GB (estimado). El primer run
  (`dinov2_l` `all`) es el de más memoria, así que un fallo de memoria sale al principio.
- Qwen3-VL-4B: `qwen.ipynb` (pod). Los VLMs por API: `api.ipynb` (laptop).
- Resultados en `notebooks/results_general.ipynb` ("Modelos de referencia").


## Parámetros

In [ ]:
import os

FULL = True          # False: piloto 32/8 para pruebas rápidas
EPOCHS = 80
BATCH = 32
WORKERS = min(15, os.cpu_count()-2)  # procesos que decodifican las imágenes

FULL_FLAG = "--full" if FULL else ""
SPLIT = "full" if FULL else "pilot"

## Preparación del entorno

In [ ]:
import os
import sys
from pathlib import Path

# The volume is mounted at /workspace only on RunPod.
RUNPOD = Path("/workspace").is_dir()
if RUNPOD:
    # SSH sessions (VS Code Remote) do not inherit the pod's variables, such as the W&B secret.
    for variable in Path("/proc/1/environ").read_text().split("\0"):
        name, _, value = variable.partition("=")
        if name == "WANDB_API_KEY":
            os.environ.setdefault(name, value)
    # The template points the Hugging Face cache at the volume, where its symlinks break
    # (object storage); keep the pretrained weights on the pod's local disk instead.
    os.environ["HF_HOME"] = "/root/.cache/huggingface"

REPOSITORY_URL = "https://github.com/victorabellotorres/thesis-multimodal-health-ai.git"
# Local disk of the pod: fast, but lost when the pod ends.
REPOSITORY_DIR = Path("/root/thesis-multimodal-health-ai")
BRANCH = "final-experiments"
# Persistent storage: the volume mounted at /workspace.
PERSIST_ROOT = Path("/workspace/nutrition5k")

if RUNPOD:
    if (REPOSITORY_DIR / ".git").is_dir():
        !git -C {REPOSITORY_DIR} fetch
        !git -C {REPOSITORY_DIR} checkout {BRANCH}
        !git -C {REPOSITORY_DIR} pull --ff-only
    else:
        !git clone -b {BRANCH} {REPOSITORY_URL} {REPOSITORY_DIR}
    %cd {REPOSITORY_DIR}
    # The image's system Python is "externally managed" (PEP 668); the pod is disposable.
    %pip install -q --break-system-packages -e .[train]
elif Path.cwd().parent.name == "notebooks":
    # Local: the package is already installed in code/.venv; run from the repository root.
    os.chdir("../..")

# The kernel's interpreter, so `!` commands use the same environment locally (VS Code may not activate .venv).
PYTHON = sys.executable

if RUNPOD:
    # Results live on the volume so nothing is lost when the pod ends.
    (PERSIST_ROOT / "outputs").mkdir(parents=True, exist_ok=True)
    if not Path("outputs").exists():
        Path("outputs").symlink_to(PERSIST_ROOT / "outputs")
print("W&B:", "on" if os.environ.get("WANDB_API_KEY") else "off (only local files)")

In [ ]:
DATA_TAR = PERSIST_ROOT / f"data-{SPLIT}-overhead.tar"

if RUNPOD and DATA_TAR.is_file():
    !tar -xf {DATA_TAR}
else:
    !{PYTHON} -m nutrition5k fetch
    !{PYTHON} -m nutrition5k prepare {FULL_FLAG}
    if RUNPOD and _exit_code == 0:  # a failed prepare must not be cached as a partial .tar
        !tar -cf {DATA_TAR} data
!{PYTHON} -m nutrition5k preprocess  # excluded_dishes.csv and dish_groups.csv
!{PYTHON} -m nutrition5k check {FULL_FLAG}

In [ ]:
import subprocess

# The common recipe; train(..., **changes) replaces some of its values.
RECIPE = dict(heads="single", hidden=(512,), image_size=224, backbone_lr=1e-4, head_width=256, augment="d4",
              schedule="cosine", weight_decay=0.01, drop_path=0.0)


def train(backbone, unfreeze="all", pooling=None, seed=0, **changes):
    """One run of the common recipe with ``changes``; finished runs are skipped, interrupted ones resume.

    Stops the notebook if CUDA is gone (a host fault once made `train` fall back to CPU, 2026-10-28) or if
    the run fails.
    """
    if RUNPOD and subprocess.run([PYTHON, "-c", "import sys, torch; sys.exit(not torch.cuda.is_available())"]).returncode:
        raise RuntimeError("CUDA is not available on this pod")
    options = RECIPE | changes
    flags = " ".join(f"--hidden {size}" for size in options.pop("hidden"))
    flags += "".join(f" --{name.replace('_', '-')} {value}" for name, value in options.items())
    command = (
        f"{PYTHON} -m nutrition5k train --backbone {backbone} --unfreeze {unfreeze} {flags} "
        f"--epochs {EPOCHS} --batch-size {BATCH} --num-workers {WORKERS} --seed {seed} {FULL_FLAG}"
    )
    if pooling:
        command += f" --pooling {pooling}"
    print(command)
    get_ipython().system(command)
    if get_ipython().user_ns["_exit_code"]:
        raise RuntimeError(f"train failed: {command}")


## Paso 1: DINOv2-L y DINOv3-L, semilla 0

`all`, 12, 6 y 0 bloques en cada uno: 8 runs de la receta común.


In [ ]:
for backbone in ("dinov2_l", "dinov3_l"):
    for unfreeze in ("all", 12, 6, 0):
        train(backbone, unfreeze, "cls+mean")

## Paso 2: semillas 1 y 2 del mejor nivel

Rellenar `BEST` con el mejor nivel de cada uno según la validación del paso 1 (`results_general.ipynb`). Vacío no
entrena nada.


In [ ]:
BEST = {"dinov2_l": 6, "dinov3_l": 6}  # Victor, 2026-11-05

for seed in (1, 2):
    for backbone, unfreeze in BEST.items():
        train(backbone, unfreeze, "cls+mean", seed)
